# Stage-by-stage transform-family benchmark (DWT vs SWT vs WPT)

This notebook imports the two program files in this folder and runs the benchmark one stage at a time, so you can inspect each intermediate result:

- the operating points;
- the search space and the ranking of the candidates;
- the selected configuration of each transform family;
- the processing time;
- the tables and the statistical tests.

| Stage | Function in `transform_family_benchmark.py` | Result |
|---|---|---|
| 1 | `simulate_operating_points()` | 60 operating points × 16 OFDM frames |
| 2 | `search_configurations()` | 54 configurations per family, 162 in total |
| 3 | `select_configurations()` | Configuration with the largest mean denoising gain per family |
| 4 | `measure_latency()` | Per-frame processing time of the selected configurations |
| 5 | `performance_table()`, `complexity_table()`, `friedman_table()`, `posthoc_table()` | Tables 6, 7 and 8, and the pairwise tests quoted in the text |

**How to run.**

1. Open this folder in Visual Studio Code.
2. Open this notebook and choose the project's Python environment with **Select Kernel** (top right).
3. Click **Run All**. The whole notebook takes about 1–2 minutes; stage 2 takes most of it.

Run the cells in order. The notebook calls the same functions as the script, with the same settings and seeds, so it produces the same numbers.

The processing times are wall-clock measurements, so they depend on the computer. Every other value is deterministic.

## Setup

In [12]:
import platform
import sys
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "transform_benchmark_core.py").exists():
    raise FileNotFoundError("Start the notebook from the folder that contains transform_benchmark_core.py.")
sys.path.insert(0, str(ROOT))

import transform_benchmark_core as core
import transform_family_benchmark as bench

print("Python    ", platform.python_version())
for pkg in ["numpy", "scipy", "pandas", "pywavelets"]:
    print(f"{pkg:<10}", version(pkg))

Python     3.11.9
numpy      2.4.6
scipy      1.17.1
pandas     3.0.6
pywavelets 1.9.0


---
## 1. Operating points

Each frame carries 14 OFDM symbols:

- 64-point FFT with a 16-sample cyclic prefix, giving 1120 complex samples per frame;
- 48 active subcarriers carrying QPSK;
- the waveform is normalised to unit average power.

Middleton Class-A noise is added at every combination of:

- Eb/N0: 6, 10 and 14 dB;
- impulsive index A: 0.005 and 0.02;
- Gaussian-to-impulsive power ratio T: 0.05 and 0.20;


In [13]:
points = bench.simulate_operating_points()
n_samples = points[0].tx.shape[1]
print(f"{len(points)} operating points x {points[0].tx.shape[0]} frames, {n_samples} complex samples per frame, "
      f"{points[0].bits.shape[1]} bits per frame")
overview = pd.DataFrame([{"Eb/N0 (dB)": p.ebn0_db, "A": p.A, "T": p.T, "mean input SNR (dB)": p.snr_in_db.mean()}
                         for p in points])
overview.groupby(["Eb/N0 (dB)", "A", "T"]).agg(seeds=("mean input SNR (dB)", "size"), mean_input_SNR_dB=("mean input SNR (dB)", "mean")).round(2)

60 operating points x 16 frames, 1120 complex samples per frame, 1344 bits per frame


seeds  mean_input_SNR_dB
Eb/N0 (dB) A     T                             
6.0        0.005 0.05      5               7.42
                 0.20      5               7.21
           0.020 0.05      5               6.84
                 0.20      5               6.81
10.0       0.005 0.05      5              11.42
                 0.20      5              11.21
           0.020 0.05      5              10.84
                 0.20      5              10.81
14.0       0.005 0.05      5              15.42
                 0.20      5              15.21
           0.020 0.05      5              14.84
                 0.20      5              14.81

---
## 2. Search space

The search space for each transform family is every combination of:

- mother wavelet: db4, sym4, coif4;
- decomposition level J: 2, 3;
- threshold function: scaled MAD (2.5 σ), universal, BayesShrink;
- threshold rule: soft, hard, garrote.

That gives 54 configurations per family.

For every configuration, the in-phase and quadrature components are processed separately. In each detail subband:

1. The noise level is estimated as σ = median(|c|)/0.6745.
2. The coefficients at or above the threshold are taken as the impulse estimate, shaped by the threshold rule.
3. The impulse estimate is reconstructed and subtracted from the received waveform.

Every configuration is applied to all 960 frames. This stage takes about one minute.

In [14]:
print({t: len(bench.candidate_grid(t)) for t in bench.TRANSFORMS}, "configurations per family")
search = bench.search_configurations(points)
print("evaluated:", {t: len(r) for t, r in search.items()})

{'DWT': 54, 'SWT': 54, 'WPT': 54} configurations per family
evaluated: {'DWT': 54, 'SWT': 54, 'WPT': 54}


---
## 3. Selection

The selected configuration of each family is the one with the largest mean denoising gain over the 60 operating points. Ties would be broken by the lower FLOP proxy; none occur. Below are the five best-ranked candidates of each family. Rank 1 is the selected configuration.

In [15]:
selected = bench.select_configurations(search, n_samples)
for t, (cfg, _) in selected.items():
    print(f"[{t}] selected: {cfg.label()}")
    display(bench.candidate_ranking(search, t).head(5).round({"ΔSNR (dB)": 3, "NMSE": 4, "BER": 5, "EVM (%)": 3}))

[DWT] selected: DWT/sym4/J=3/scaled_mad x2.5/garrote


,configuration,ΔSNR (dB),NMSE,BER,EVM (%)
rank,,,,,
1,DWT/sym4/J=3/scaled_mad x2.5/garrote,1.088,0.0744,0.00298,22.359
2,DWT/sym4/J=3/scaled_mad x2.5/soft,1.042,0.0785,0.00328,22.383
3,DWT/sym4/J=2/scaled_mad x2.5/garrote,1.010,0.0772,0.00334,22.568
4,DWT/sym4/J=2/scaled_mad x2.5/soft,0.955,0.0810,0.00362,22.615
5,DWT/coif4/J=3/scaled_mad x2.5/garrote,0.946,0.0772,0.00339,22.760


[SWT] selected: SWT/sym4/J=3/scaled_mad x2.5/garrote


,configuration,ΔSNR (dB),NMSE,BER,EVM (%)
rank,,,,,
1,SWT/sym4/J=3/scaled_mad x2.5/garrote,1.644,0.0660,0.00196,21.017
2,SWT/sym4/J=3/universal/hard,1.530,0.0689,0.00209,21.071
3,SWT/sym4/J=2/scaled_mad x2.5/garrote,1.482,0.0694,0.00235,21.432
4,SWT/coif4/J=3/scaled_mad x2.5/garrote,1.472,0.0691,0.00233,21.490
5,SWT/db4/J=3/scaled_mad x2.5/garrote,1.439,0.0696,0.00235,21.583


[WPT] selected: WPT/sym4/J=2/scaled_mad x2.5/garrote


,configuration,ΔSNR (dB),NMSE,BER,EVM (%)
rank,,,,,
1,WPT/sym4/J=2/scaled_mad x2.5/garrote,0.991,0.0749,0.00278,22.669
2,WPT/sym4/J=2/scaled_mad x2.5/soft,0.980,0.0789,0.00313,22.549
3,WPT/coif4/J=2/scaled_mad x2.5/soft,0.863,0.0816,0.00348,22.866
4,WPT/coif4/J=2/scaled_mad x2.5/garrote,0.862,0.0778,0.00311,23.022
5,WPT/sym4/J=2/universal/garrote,0.841,0.0833,0.00349,22.739


---
## 4. Processing time

Each frame of every operating point is processed separately by each selected configuration. A frame's time is the median of 3 calls, measured after 3 warm-up calls. This takes a few seconds.

These are wall-clock times on *this* computer. 

In [16]:
latency = bench.measure_latency(points, selected)
op_table = bench.operating_point_table(points, selected, latency)
print("paired observations:", op_table.shape[0], "(3 transforms x 60 operating points)")
op_table.head()

paired observations: 180 (3 transforms x 60 operating points)


,transform,configuration,seed,ebn0_db,A,T,gain_db,nmse,ber,evm_percent,time_ms
0,DWT,DWT/sym4/J=3/scaled_mad x2.5/garrote,101,6.0,0.005,0.05,3.807824,0.095869,0.008557,26.571131,0.685600
1,DWT,DWT/sym4/J=3/scaled_mad x2.5/garrote,202,6.0,0.005,0.05,3.306447,0.088843,0.007440,25.386869,0.481569
2,DWT,DWT/sym4/J=3/scaled_mad x2.5/garrote,303,6.0,0.005,0.05,3.321663,0.089184,0.006278,24.727473,0.254512
3,DWT,DWT/sym4/J=3/scaled_mad x2.5/garrote,404,6.0,0.005,0.05,3.600634,0.094205,0.008836,27.176446,0.276287
4,DWT,DWT/sym4/J=3/scaled_mad x2.5/garrote,505,6.0,0.005,0.05,3.206512,0.079273,0.005906,23.493328,0.360962


---
## 5. Tables

### Receiver-level denoising performance

Each entry is the mean ± the half-width of the 95 % confidence interval (1.96 · SD/√60) over the 60 operating points.

In [17]:
perf, table6 = bench.performance_table(op_table)
table6

,Method,ΔSNR / denoising gain,NMSE,BER outcome,EVM
0,DWT,1.088 ± 0.291 dB,(7.437 ± 1.060) × 10^-2,(2.982 ± 0.848) × 10^-3,22.359 ± 1.582 %
1,SWT,1.644 ± 0.307 dB,(6.597 ± 0.962) × 10^-2,(1.962 ± 0.578) × 10^-3,21.017 ± 1.538 %
2,WPT,0.991 ± 0.332 dB,(7.492 ± 1.050) × 10^-2,(2.781 ± 0.793) × 10^-3,22.669 ± 1.530 %


### Computational complexity

The minimum frame budget T_frame is, by definition, equal to the measured mean time. The FLOP proxy is deterministic:

- DWT: 4 L_f Σ_{j=1..J} n/2^(j−1);
- SWT and WPT: 8 L_f n J;

where n = 1120 and L_f = 8 for sym4.

In [ ]:
comp, table7 = bench.complexity_table(op_table, selected, n_samples)
table7

### Friedman tests across DWT, SWT and WPT

The tests use the 60 matched operating points. MSE equals NMSE because the transmitted waveform has unit average power.

In [ ]:
fried, table8 = bench.friedman_table(op_table)
table8

### Pairwise post-hoc tests

These are Wilcoxon signed-rank tests, Holm-corrected over the 3 pairs of each metric.

In [ ]:
posthoc = bench.posthoc_table(op_table)
posthoc.pivot(index="Comparison", columns="Metric", values="p (Holm)")[list(bench.METRICS.values())] \
       .apply(lambda c: c.map("{:.1e}".format))

In [ ]:
out = ROOT / "results"
out.mkdir(exist_ok=True)
perf.to_csv(out / "transform_performance.csv", index=False)
comp.to_csv(out / "transform_complexity.csv", index=False)
fried.to_csv(out / "transform_friedman.csv", index=False)
posthoc.to_csv(out / "transform_posthoc.csv", index=False)
op_table.to_csv(out / "transform_operating_points.csv", index=False)
print("Saved:", *sorted(p.name for p in out.glob("transform_*.csv")), sep="\n  ")